## 5 - Loading Processed Data

In this step, we load processed video dataset from the previously notebook


In [ ]:
import cv2
import glob
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
from pathlib import Path
from PIL import Image
import random
from sklearn.model_selection import train_test_split
import torch
from tqdm import tqdm
import urllib.request

In [ ]:
print("--- LOADING PROCESSED DATA ---")

load_path = "./processed_images/celeb_videos.csv"
celeb_video = pd.read_csv(load_path)

print(f"{len(celeb_video)} videos loaded!")
pd.set_option('display.max_colwidth', None)
display(celeb_video.sample(15))

## 6 - Cross-Dataset Test Split

In this section, we prepare the Celeb-DF-v3 dataset for **Cross-Dataset Evaluation**. 

Since we are using this entire dataset strictly to test the generalization capabilities of a model trained on a different dataset, we do not need to perform a traditional Train/Validation/Test split. Instead, we assign **100%** of the data to the **Test** split. 

This ensures complete isolation from whatever training data the model has seen previously.

In [ ]:
celeb_video['split'] = 'test'
celeb_video.sample(10)
print(celeb_video['split'].value_counts())
print(f"\nAll 'test'? {(celeb_video['split'] == 'test').all()}")

## 7 - Dataset Balancing & Frame Extraction

Before extracting frames, we first ensure a fair evaluation by creating a **perfectly balanced subset**. We match the number of Fake videos to the total number of Real videos (e.g., 890 vs 890), using stratified sampling to maintain an even distribution across all deepfake manipulation methods.

To prepare the dataset for testing, we then extract **1 representative frame** per video (`fpv=1`) and process them through our targeted pipeline:

* **SSD Face Detection**: We use OpenCV's DNN (ResNet-10 SSD) to detect faces. Detections are cropped with a 15% margin and padded into a square to prevent aspect-ratio distortion.
* **Fallback Method:** If the model fails to confidently detect a face, we automatically apply a static center crop.
* **Standardization**: Every extracted face is resized to **224x224 pixels**, which is the standard input size for most modern vision backbones.

In [ ]:
df_real = celeb_video[celeb_video['label'] == 0].copy()

df_fake_pool = celeb_video[celeb_video['label'] == 1]
n_samples = len(df_real) # 890

df_fake_balanced = df_fake_pool.groupby('method', group_keys=False).apply(
    lambda x: x.sample(n=min(len(x), n_samples // df_fake_pool['method'].nunique() + 1), random_state=42)
)

if len(df_fake_balanced) > n_samples:
    df_fake_balanced = df_fake_balanced.sample(n_samples, random_state=42)
elif len(df_fake_balanced) < n_samples:
    diff = n_samples - len(df_fake_balanced)
    extra = df_fake_pool.drop(df_fake_balanced.index).sample(diff, random_state=42)
    df_fake_balanced = pd.concat([df_fake_balanced, extra])

df_test_sampling = pd.concat([df_real, df_fake_balanced]).sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Subset bilanciato creato: {len(df_test_sampling)} video.")
print(df_test_sampling['label'].value_counts())

In [ ]:
# CONFIGURATION
base_root = "CelebDF_Test"
DETECTOR_DIR = Path.cwd().parent
fpv = 1 
target_size = (224, 224)

print(f"STARTING EXTRACTION: {len(df_test_sampling)} videos | {fpv} frames each")

os.makedirs(base_root, exist_ok=True)
for label_folder in ["original", "fake"]:
    os.makedirs(os.path.join(base_root, label_folder), exist_ok=True)

# Setup Models Directory
model_dir = DETECTOR_DIR / "Utility" / "FaceDetection"
model_dir.mkdir(parents=True, exist_ok=True)

prototxt_path = str(model_dir / "deploy.prototxt")
model_path = str(model_dir / "res10_300x300_ssd_iter_140000.caffemodel")

# Download models if they don't exist
if not os.path.exists(prototxt_path):
    print("Downloading deploy.prototxt...")
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/opencv/opencv/master/samples/dnn/face_detector/deploy.prototxt",
        prototxt_path
    )

if not os.path.exists(model_path):
    print("Downloading caffemodel (circa 10 MB)...")
    urllib.request.urlretrieve(
        "https://github.com/opencv/opencv_3rdparty/raw/dnn_samples_face_detector_20170830/res10_300x300_ssd_iter_140000.caffemodel",
        model_path
    )

# Load SSD (Face Detector)
net = cv2.dnn.readNetFromCaffe(prototxt_path, model_path)
net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
net.setPreferableTarget(cv2.dnn.DNN_TARGET_CPU)

# Utility
def apply_static_center_crop(frame, target_size):
    h, w = frame.shape[:2]
    min_dim = min(h, w)
    start_x = (w - min_dim) // 2
    start_y = (h - min_dim) // 2
    crop_frame = frame[start_y:start_y+min_dim, start_x:start_x+min_dim]
    return cv2.resize(crop_frame, target_size, interpolation=cv2.INTER_AREA)

# MAIN LOOP 
extracted_metadata = []

for idx, row in tqdm(df_test_sampling.iterrows(), total=len(df_test_sampling)):
    video_path = row['full_path']
    label_str = "original" if row['label'] == 0 else "fake"
    
    base_name = f"{row['target']}_{row['source']}_{row['method']}_{row['video'].replace('.mp4','')}"
    save_dir = os.path.join(base_root, label_str)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        continue

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames < fpv:
        cap.release()
        continue

    target_indices = np.linspace(0, total_frames - 1, fpv, dtype=int)
    
    for count, target_idx in enumerate(target_indices):
        cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
        ret, frame = cap.read()
        if not ret: 
            continue

        processed_frame = None
        strategy = "cc" 

        # Face Detection via SSD
        (h, w) = frame.shape[:2]
        blob = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0))
        net.setInput(blob)
        detections = net.forward()

        if detections.shape[2] > 0:
            confidence = detections[0, 0, 0, 2]
            if confidence > 0.6:
                box = detections[0, 0, 0, 3:7] * np.array([w, h, w, h])
                x1, y1, x2, y2 = box.astype(int)

                w_b, h_b = x2 - x1, y2 - y1
                margin = int(max(w_b, h_b) * 0.15)
                cx1, cy1 = max(0, x1-margin), max(0, y1-margin)
                cx2, cy2 = min(w, x2+margin), min(h, y2+margin)
                
                crop_face = frame[cy1:cy2, cx1:cx2]
                if crop_face.size > 0:
                    fh, fw = crop_face.shape[:2]
                    side = max(fh, fw)
                    square = np.zeros((side, side, 3), np.uint8)
                    square[(side-fh)//2 : (side-fh)//2+fh, (side-fw)//2 : (side-fw)//2+fw] = crop_face
                    processed_frame = cv2.resize(square, target_size, interpolation=cv2.INTER_AREA)
                    strategy = "ssd"

        # Fallback if SSD fails
        if processed_frame is None:
            processed_frame = apply_static_center_crop(frame, target_size)

        # Saving Process
        img_name = f"{base_name}_f{count}_{strategy}.jpg"
        save_path = os.path.join(save_dir, img_name)
        cv2.imwrite(save_path, processed_frame, [cv2.IMWRITE_JPEG_QUALITY, 95])
        
        extracted_metadata.append({
            'path': save_path,
            'label': row['label'],
            'split': 'test',
            'dataset': "CelebDF",
            'method': row.get('method', 'NA')
        })

    cap.release()

celeb_frames = pd.DataFrame(extracted_metadata)

print(f"\n--- EXTRACTION COMPLETE: {len(celeb_frames)} frames saved ---")
if not celeb_frames.empty:
    display(celeb_frames.sample(min(15, len(celeb_frames))))

## 8 - Visual Sanity Check 

This serves as a critical "sanity check" to verify that the frame extraction process completed successfully without generating corrupted or blank images

In [ ]:
base_root_celeb = "CelebDF_Test" 

print("--- CELEB-DF-V3 SANITY CHECK (DATAFRAME vs DISK) ---")

expected_celeb = len(celeb_frames)

def count_celeb_frames(base_dir):
    if not os.path.exists(base_dir):
        return 0, 0, 0
    total, ssd, cc = 0, 0, 0
    for root, dirs, files in os.walk(base_dir):
        for file in files:
            if file.endswith('.jpg') or file.endswith('.png'):
                total += 1
                if '_ssd' in file:
                    ssd += 1
                elif '_cc' in file:
                    cc += 1
    return total, ssd, cc

# Execute physics count on HD
actual_celeb, celeb_ssd, celeb_cc = count_celeb_frames(base_root_celeb)

# Print stathistics 
print(f"\n--- GLOBAL CELEB-DF-V3 STATISTICS ---")
print(f"Frames in DataFrame (Expected): {expected_celeb}")
print(f"Frames on Disk (Found):         {actual_celeb}")

if expected_celeb == actual_celeb and expected_celeb > 0:
    print("STATUS: PERFECT! DataFrame and Disk are perfectly synced.")
elif actual_celeb == 0:
    print("STATUS: WARNING! No frames found on disk. Check the path.")
else:
    print(f"STATUS: MISMATCH! Difference of {abs(expected_celeb - actual_celeb)} frames.")

if actual_celeb > 0:
    print("-" * 35)
    print(f"Extraction Quality Check:")
    print(f"  -> SSD Detected Faces: {celeb_ssd} ({(celeb_ssd/actual_celeb)*100:.1f}%)")
    print(f"  -> Center Crop Fallback: {celeb_cc} ({(celeb_cc/actual_celeb)*100:.1f}%)")
    
    # Check extra: folders balance
    path_real = os.path.join(base_root_celeb, "original")
    path_fake = os.path.join(base_root_celeb, "fake")
    
    n_real = len([f for f in os.listdir(path_real) if f.endswith('.jpg')]) if os.path.exists(path_real) else 0
    n_fake = len([f for f in os.listdir(path_fake) if f.endswith('.jpg')]) if os.path.exists(path_fake) else 0
    
    print("-" * 35)
    print(f"Class Balance Check on Disk:")
    print(f"  -> Original (Real): {n_real}")
    print(f"  -> Fake:            {n_fake}")

print(f"\nAudit completed for Celeb-DF-v3.")

In [ ]:
def show_celeb_frames_from_df(dataframe, num_images=10):
    print("\n--- VISUALIZING RANDOM FRAMES FROM CELEB-DF-V3 ---")
    
    sample = dataframe.sample(n=min(num_images, len(dataframe))).to_dict('records')
    
    if not sample:
        print("Nessuna immagine trovata nel DataFrame...")
        return 

    cols = 5
    rows = math.ceil(len(sample) / cols)
    
    plt.figure(figsize=(20, 5 * rows))
    plt.suptitle("Celeb-DF-v3 Test Samples (224x224)", fontsize=22, fontweight='bold', y=1.02)

    for i, row in enumerate(sample):
        plt.subplot(rows, cols, i+1)

        img_path = row['path']
        try:
            img = Image.open(img_path)
            plt.imshow(img)
        except Exception as e:
            plt.text(0.5, 0.5, 'Errore Caricamento', ha='center', va='center')
            
        plt.axis('off')

        label_text = "REAL" if row['label'] == 0 else "FAKE"
        method_name = str(row['method']).upper()
        
        strategy = "SSD" if "_ssd" in img_path else "CC"
        
        title_color = 'green' if label_text == "REAL" else 'red'
        plt.title(f"METHOD: {method_name}\nL: {label_text} | {strategy}", 
                  color=title_color, fontsize=11, fontweight='bold', pad=10)

    plt.tight_layout()
    plt.show()

show_celeb_frames_from_df(celeb_frames, num_images=10)

## 8 - Saving Processed Data

To conclude this notebook, we save the fully cleaned and processed DataFrame to a local CSV file (`./processed_images/celeb_frames.csv`). This ensures our prepared dataset is safely stored and ready to be directly loaded into the next notebook of our pipeline without needing to re-run the preprocessing steps.

In [ ]:
print("--- SAVING DATAFRAME ---")

output_dir = "./processed_images"
os.makedirs(output_dir, exist_ok=True)

save_path = os.path.join(output_dir, "celeb_frames.csv")
celeb_frames.to_csv(save_path, index=False)

print(f"Data succesfully saved to: {save_path}")